# One million increments, and the counter says 712 431

`counter += 1` is three steps, read, add and write, and threads interleave
between them. The notebook runs a smaller version, 20 000 increments over four
threads. Python's interpreter lock hides the race for tiny loops, so we put an
explicit "let someone else run" between the read and the write: that is what
the hardware does on its own, uninvited, in any other language.

In [1]:
import threading, time

counter = 0
def worker(n):
    global counter
    for _ in range(n):
        value = counter          # read
        time.sleep(0)            # let another thread run here: what the hardware does on its own, uninvited
        counter = value + 1      # add, write

threads = [threading.Thread(target=worker, args=(5_000,)) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print("expected 20,000   got", f"{counter:,}")

expected 20,000   got 5,000


Most of the increments vanished: each thread read the counter, let the others
run, and then wrote back a value that ignored everything they had done in the
meantime. Without the explicit yield the loss is smaller and random, and in C,
Java or Go it happens with no help at all.

## Fix 1: a lock makes the three steps one step

In [2]:
counter, lock = 0, threading.Lock()
def worker_locked(n):
    global counter
    for _ in range(n):
        with lock:                # only one thread inside at a time
            counter += 1

threads = [threading.Thread(target=worker_locked, args=(5_000,)) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print("with a lock:", f"{counter:,}")

with a lock: 20,000


Correct, and the locked section is now *serial*, which Amdahl's law will charge
you for.

## Fix 2: no shared state at all

In [3]:
def worker_private(n, results, i):
    local = 0
    for _ in range(n):
        local += 1                # nothing shared
    results[i] = local

results = [0] * 4
threads = [threading.Thread(target=worker_private, args=(5_000, results, i)) for i in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print("per-thread counts:", results, "  sum:", f"{sum(results):,}")

per-thread counts: [5000, 5000, 5000, 5000]   sum: 20,000


Each thread keeps its own count; the four are added once at the end. That is
the Map and Reduce of MapReduce, and it is why the framework has no race.

**What we used:** threads sharing memory, interleaving, locks, and why MapReduce
keeps state local (session 12, sections 4, 5, 9 and 11).